## Here we stack maps in order to fill the holes due to the scanning strategy before fitting the order 0 of the Moon that will give us the line of sight of each detector

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2

In [ ]:
%config InlineBackend.figure_format='retina'
from IPython import display
display.display(display.HTML("<style>.container { width:95% !important; }</style>"))

# to have interactive plots : pip install ipympl
%matplotlib ipympl

### General imports
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pickle
import fitsio
from scipy.interpolate import LinearNDInterpolator
from scipy.optimize import least_squares
from mpl_toolkits.axes_grid1 import make_axes_locatable

from scipy.interpolate import interp1d

# import pysimulators
# this should allow us to have high resolution maps of the Moon
import healsparse as hsp

plt.rc('figure',figsize=(10,10))
plt.rc('font',size=16)

from datetime import datetime
### Astropy configuration
import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
from astropy.stats import sigma_clip

In [ ]:
ang_size = 40 # degrees
xsize = 201
reso = ang_size*60/xsize # arcmin/pix
min_plot = -5e3
max_plot = 1.2e4
center = [0, 90]

In [ ]:
datadir_MoonProject = "/Users/huchet/Documents/code/scripts/MoonProject/"

In [ ]:
list_of_observations = [                  "2026-07-23_rise",
                        "2026-07-24_set", "2026-07-24_rise",
                        "2026-07-25_set", "2026-07-25_rise",
                        "2026-07-26_set", "2026-07-26_rise",
                        "2026-07-27_set", "2026-07-27_rise",
                        "2026-07-28_set"]

dict_already_computed_tshifts = {                         "2026-07-23_rise": -0.63,
                                 "2026-07-24_set": -0.65, "2026-07-24_rise": -0.63,
                                 "2026-07-25_set": -0.7,  "2026-07-25_rise": -0.6,
                                 "2026-07-26_set": -0.52, "2026-07-26_rise": -0.7,
                                 "2026-07-27_set": -0.65, "2026-07-27_rise": -0.85,
                                 "2026-07-28_set": -0.75}

In [ ]:
TESNum = 218 #96
i_TES = TESNum - 1
nside = 512
allmaps_chosen_TES = []

for i_obs, obsname in enumerate(list_of_observations):
    ObsDate, scanning = obsname.split("_")
    tshift = dict_already_computed_tshifts[obsname]
    file_name = "allmaps-{}_{}_nside{}_zenith_tshift{}.pkl".format(ObsDate, scanning, nside, tshift)
    allTESNum, allmaps = pickle.load( open( datadir_MoonProject + file_name, "rb" ) )
    allmaps_chosen_TES.append(allmaps[i_TES])
    print("Read {}".format(file_name))
allmaps_chosen_TES = np.array([map_ for map_ in allmaps_chosen_TES])

In [ ]:
allmaps_chosen_TES_masked = np.ma.masked_array(allmaps_chosen_TES, allmaps_chosen_TES == hp.UNSEEN)
stacked_map_chosen_TES = np.mean(allmaps_chosen_TES_masked, axis=0)

In [ ]:
print(np.shape(stacked_map_chosen_TES))

In [ ]:
for i_obs, obsname in enumerate(list_of_observations):
    hp.gnomview(allmaps_chosen_TES[i_obs], reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center,return_projected_map=False, no_plot=False)
    plt.show()

In [ ]:
mapsb_proj = hp.gnomview(stacked_map_chosen_TES, reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center, return_projected_map=True, no_plot=True)
mapsb_proj[mapsb_proj > 1e10] = 0

In [ ]:
print(np.shape(allmaps_chosen_TES))

In [ ]:
fig, ax = plt.subplots()
ax.imshow(mapsb_proj, vmin=min_plot, vmax=max_plot)
plt.show()

In [ ]:
allmaps_chosen_TES_masked_ = sigma_clip(allmaps_chosen_TES_masked, sigma=3, axis=0)
stacked_map_chosen_TES_ = np.mean(allmaps_chosen_TES_masked_, axis=0)

In [ ]:
print(np.shape(allmaps_chosen_TES_masked_.mask))

In [ ]:
# for i in range(len(list_of_observations)):
i = 1
hp.gnomview(allmaps_chosen_TES_masked_.mask[i], reso=reso, xsize=xsize,
                    rot=center,return_projected_map=False, no_plot=False)
plt.show()
hp.gnomview((allmaps_chosen_TES == hp.UNSEEN)[i], reso=reso, xsize=xsize,
                    rot=center,return_projected_map=False, no_plot=False)
plt.show()
hp.gnomview(np.logical_and((allmaps_chosen_TES == hp.UNSEEN)[i] allmaps_chosen_TES_masked_.mask[i]), reso=reso, xsize=xsize,
                    rot=center,return_projected_map=False, no_plot=False)

In [ ]:
mapsb_proj_ = hp.gnomview(stacked_map_chosen_TES_, reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center, return_projected_map=True, no_plot=True)
mapsb_proj_[mapsb_proj_ == 1e20] = 0

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(mapsb_proj, vmin=min_plot, vmax=max_plot)
axs[1].imshow(mapsb_proj_, vmin=min_plot, vmax=max_plot)
axs[2].imshow(mapsb_proj_ - mapsb_proj, vmin=min_plot, vmax=max_plot)
plt.show()

In [ ]:
sigma_ok = np.std(allmaps_chosen_TES, where=allmaps_chosen_TES != hp.UNSEEN, axis=0)

In [ ]:
sigma_proj = hp.gnomview(sigma_ok, reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center, return_projected_map=True, no_plot=True)
fig, ax = plt.subplots()
ax.imshow(sigma_proj, vmin=min_plot, vmax=max_plot)
plt.show()